In [2]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from pathlib import Path


In [3]:
DICOM_BASE = "/mnt/cafetera/mammo/vindr/images"
ANNOTATIONS = "/home/enric_sena/Desktop/prova_enric/vindr_dataset/finding_annotations.csv"

# Load annotations, preview them, and choose one annotated image for testing.
annotations = pd.read_csv(ANNOTATIONS)
df=annotations # per practicitat més endavant
df.head()

,study_id,series_id,image_id,laterality,view_position,height,width,breast_birads,breast_density,finding_categories,finding_birads,xmin,ymin,xmax,ymax,split
0,48575a27b7c992427041a82fa750d3fa,26de4993fa6b8ae50a91c8baf49b92b0,4e3a578fe535ea4f5258d3f7f4419db8,R,CC,3518,2800,BI-RADS 4,DENSITY C,['Mass'],BI-RADS 4,2355.139893,1731.640015,2482.979980,1852.750000,training
1,48575a27b7c992427041a82fa750d3fa,26de4993fa6b8ae50a91c8baf49b92b0,dac39351b0f3a8c670b7f8dc88029364,R,MLO,3518,2800,BI-RADS 4,DENSITY C,['Mass'],BI-RADS 4,2386.679932,1240.609985,2501.800049,1354.040039,training
2,75e8e48933289d70b407379a564f8594,853b70e7e6f39133497909d9ca4c756d,c83f780904f25eacb44e9030f32c66e1,R,CC,3518,2800,BI-RADS 3,DENSITY C,['Global Asymmetry'],BI-RADS 3,2279.179932,1166.510010,2704.439941,2184.260010,training
3,75e8e48933289d70b407379a564f8594,853b70e7e6f39133497909d9ca4c756d,893528bc38a0362928a89364f1b692fd,R,MLO,3518,2800,BI-RADS 3,DENSITY C,['Global Asymmetry'],BI-RADS 3,1954.270020,1443.640015,2589.760010,2193.810059,training
4,c3487424fee1bdd4515b72dc3fd69813,77619c914263eae44e9099f1ce07192c,318264c881bf12f2c1efe5f93920cc37,R,CC,3518,2800,BI-RADS 4,DENSITY C,['Architectural Distortion'],BI-RADS 4,2172.300049,1967.410034,2388.699951,2147.159912,training


In [4]:
df['split'].value_counts()

split
training    16391
test         4095
Name: count, dtype: int64

In [5]:
df['study_id'].groupby(df['split']).nunique()

split
test        1000
training    4000
Name: study_id, dtype: int64

In [6]:
#df['finding_categories'].value_counts()

df["is_positive"] = (df["finding_categories"]!="['No Finding']").astype(int)
print(df["is_positive"].value_counts())



is_positive
0    18232
1     2254
Name: count, dtype: int64


In [7]:
df["study_positive"]=df.groupby("study_id")["is_positive"].transform("max")

In [8]:
training = df[df["split"]=="training"]
print(training.groupby("study_id")["study_positive"].first().value_counts())

study_positive
0    3263
1     737
Name: count, dtype: int64


In [9]:
study_df=(
    training[["study_id","study_positive"]]
    .drop_duplicates("study_id")
)

train_studies, val_studies = train_test_split(
    study_df,
    test_size=0.2,
    random_state=42,
    stratify=study_df["study_positive"]
)

print("Train Studies:", len(train_studies))
print("Validation Studies:", len(val_studies))

print("\n Train:")
print(train_studies["study_positive"].value_counts())
print("\n Validation:")
print(val_studies["study_positive"].value_counts())


Train Studies: 3200
Validation Studies: 800

 Train:
study_positive
0    2610
1     590
Name: count, dtype: int64

 Validation:
study_positive
0    653
1    147
Name: count, dtype: int64


In [10]:
train_ids=set(train_studies["study_id"])
val_ids=set(val_studies["study_id"])

train_df=df[
    (df["split"]=="training")&
    (df["study_id"].isin(train_ids))
].copy()

val_df = df[
(df["split"] == "training") &
(df["study_id"].isin(val_ids))
].copy()

test_df = df[
df["split"] == "test"
].copy()

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

Train images: 13115
Validation images: 3276
Test images: 4095


In [11]:
output_dir = Path("/home/enric_sena/Desktop/Mammo/Lesion Detection/Yolo v8/vindr_yolo_vit/vit_splits")
output_dir.mkdir(exist_ok=True)

train_df.to_csv(output_dir / "vindr_train.csv", index=False)
val_df.to_csv(output_dir / "vindr_val.csv", index=False)
test_df.to_csv(output_dir / "vindr_test.csv", index=False)

print(f"\nCSV guardados en: {output_dir.resolve()}")


CSV guardados en: /home/enric_sena/Desktop/Mammo/Lesion Detection/Yolo v8/vindr_yolo_vit/vit_splits


In [12]:
print("\nTrain:")
print(train_df["is_positive"].value_counts())

print("\nValidation:")
print(val_df["is_positive"].value_counts())

print("\nTest:")
print(test_df["is_positive"].value_counts())


Train:
is_positive
0    11684
1     1431
Name: count, dtype: int64

Validation:
is_positive
0    2905
1     371
Name: count, dtype: int64

Test:
is_positive
0    3643
1     452
Name: count, dtype: int64
